In [1]:
import pandas as pd
import numpy as np
import re
import spacy
import unicodedata
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Charger le modèle français de spaCy
print("Chargement du modèle spaCy français...")
nlp = spacy.load("fr_core_news_md")
print("✅ Modèle chargé!")

C:\Users\suki\AppData\Local\Temp\ipykernel_7836\4238641185.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Chargement du modèle spaCy français...
✅ Modèle chargé!


In [2]:
# Charger le dataset final qu'on a créé dans la phase précédente
df = pd.read_csv('../data/processed/reclamations_dataset_final.csv')

print(f"Nombre total de réclamations : {len(df)}")
print(f"\nColonnes : {df.columns.tolist()}")
print(f"\nAperçu :")
df.head()

Nombre total de réclamations : 1108

Colonnes : ['texte', 'categorie', 'priorite']

Aperçu :


,texte,categorie,priorite
0,les frais en double viennent la carte de chôma...,erreur_administrative,moyenne
1,contacté à plusieurs reprises l'entreprise exi...,erreur_administrative,elevee
2,déposé litige poursuite visa charge compte fai...,erreur_administrative,moyenne
3,soumettre une plainte crédit extrêmement affec...,erreur_administrative,moyenne
4,transaction de crédit à la consommation entrée...,erreur_administrative,moyenne


In [3]:
def normaliser_texte(texte):
    """
    Étape 1 : Mettre en minuscules et enlever les accents.
    Exemple : 'Bonjour, Hôtel' → 'bonjour, hotel'
    """
    texte = str(texte).lower().strip()
    # Enlever les accents
    texte = unicodedata.normalize('NFKD', texte)
    texte = ''.join([c for c in texte if not unicodedata.combining(c)])
    return texte


def nettoyer_texte(texte):
    """
    Étape 2 : Enlever la ponctuation et les chiffres.
    Exemple : 'commande #12345 cassée!' → 'commande casse'
    """
    # Remplacer la ponctuation par des espaces
    texte = re.sub(r'[^\w\s]', ' ', texte)
    # Enlever les chiffres
    texte = re.sub(r'\d+', ' ', texte)
    # Enlever les espaces multiples
    texte = re.sub(r'\s+', ' ', texte).strip()
    return texte


def lemmatiser_texte(texte):
    """
    Étape 3 : Lemmatiser (ramener les mots à leur forme de base).
    Exemple : 'commandes', 'commandé', 'commande' → tous deviennent 'commander'
    Exemple : 'livraisons', 'livraison' → 'livraison'
    
    Aussi : enlever les stopwords (mots inutiles comme 'le', 'la', 'de'...)
    """
    doc = nlp(texte)
    tokens = []
    for token in doc:
        # Garder seulement les mots significatifs
        if (not token.is_stop          # pas un stopword
            and not token.is_punct      # pas une ponctuation
            and len(token.text) > 2     # plus de 2 caractères
            and token.is_alpha):         # uniquement alphabétique
            tokens.append(token.lemma_)
    return ' '.join(tokens)


def preprocesser_complet(texte):
    """
    Pipeline complet : applique les 3 étapes.
    """
    texte = normaliser_texte(texte)
    texte = nettoyer_texte(texte)
    texte = lemmatiser_texte(texte)
    return texte


# Test sur un exemple
exemple = "Bonjour, j'ai commandé le produit #12345 le 15/03/2024 mais il est arrivé CASSÉ !!!"
print(f"Original    : {exemple}")
print(f"Normalisé   : {normaliser_texte(exemple)}")
print(f"Nettoyé     : {nettoyer_texte(normaliser_texte(exemple))}")
print(f"Final       : {preprocesser_complet(exemple)}")

Original    : Bonjour, j'ai commandé le produit #12345 le 15/03/2024 mais il est arrivé CASSÉ !!!
Normalisé   : bonjour, j'ai commande le produit #12345 le 15/03/2024 mais il est arrive casse !!!
Nettoyé     : bonjour j ai commande le produit le mais il est arrive casse
Final       : bonjour commande produit arriver casser


In [4]:
from tqdm import tqdm
tqdm.pandas()

print("Preprocessing de toutes les réclamations... (cela peut prendre 2-5 minutes)")
df['texte_clean'] = df['texte'].progress_apply(preprocesser_complet)

print("✅ Preprocessing terminé!")
df[['texte', 'texte_clean', 'categorie']].head()

Preprocessing de toutes les réclamations... (cela peut prendre 2-5 minutes)


100%|██████████████████████████████████████████████████████████████████████████████| 1108/1108 [00:32<00:00, 34.41it/s]

✅ Preprocessing terminé!


,texte,texte_clean,categorie
0,les frais en double viennent la carte de chôma...,frais double venir carte chomage original suiv...,erreur_administrative
1,contacté à plusieurs reprises l'entreprise exi...,contacter reprise entreprise exiger paiement i...,erreur_administrative
2,déposé litige poursuite visa charge compte fai...,depose litige poursuite visa charge compte cha...,erreur_administrative
3,soumettre une plainte crédit extrêmement affec...,soumettre plainte credit extremement affecter ...,erreur_administrative
4,transaction de crédit à la consommation entrée...,transaction credit consommation entree service...,erreur_administrative


In [5]:
# Comparer la longueur avant/après
df['longueur_originale'] = df['texte'].str.split().str.len()
df['longueur_clean'] = df['texte_clean'].str.split().str.len()

print(f"Longueur moyenne originale : {df['longueur_originale'].mean():.0f} mots")
print(f"Longueur moyenne après nettoyage : {df['longueur_clean'].mean():.0f} mots")
print(f"\nRéduction : {(1 - df['longueur_clean'].mean()/df['longueur_originale'].mean())*100:.0f}%")

# Enlever les textes trop courts après nettoyage (moins de 3 mots)
avant = len(df)
df = df[df['longueur_clean'] >= 3].copy()
print(f"\nTextes supprimés (trop courts) : {avant - len(df)}")
print(f"Dataset final : {len(df)} réclamations")

Longueur moyenne originale : 96 mots
Longueur moyenne après nettoyage : 66 mots

Réduction : 31%

Textes supprimés (trop courts) : 0
Dataset final : 1108 réclamations


In [6]:
# Garder seulement les colonnes utiles
df_final = df[['texte_clean', 'categorie', 'priorite']].copy()
df_final.to_csv('../data/processed/reclamations_clean.csv', index=False)

print("✅ Données nettoyées sauvegardées !")
print(f"Fichier : data/processed/reclamations_clean.csv")
print(f"Taille : {len(df_final)} réclamations")

✅ Données nettoyées sauvegardées !
Fichier : data/processed/reclamations_clean.csv
Taille : 1108 réclamations
